# Qwen LoRA DPO → verifiable coding preferences
Select a T4/L4 GPU runtime. This notebook runs real training/evaluation; no prefilled metrics.
It creates a Python 3.11 venv so Colab's default Python version does not affect the pinned stack.
Upload the source zip created by `python scripts/package_source.py`, or set `REPO_URL` below.
Full runs take time; preserve outputs before the Colab runtime expires. W&B is opt-in.


In [ ]:
import os, subprocess, pathlib, sys, zipfile
REPO_URL = ""  # Your published GitHub clone URL, or leave blank to upload the supplied source zip.
root = pathlib.Path('/content/rl-alignment-and-marl')
if not root.exists():
    if REPO_URL:
        subprocess.run(['git', 'clone', REPO_URL, str(root)], check=True)
    else:
        from google.colab import files
        uploaded = files.upload()
        name = next(n for n in uploaded if n.endswith('.zip'))
        with zipfile.ZipFile(name) as z:
            base=pathlib.Path('/content').resolve()
            for entry in z.namelist():
                if not (base / entry).resolve().is_relative_to(base):
                    raise ValueError('Unsafe zip path')
            z.extractall(base)
os.chdir(root)
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','uv==0.6.17'],check=True)
os.environ['PATH'] = str(root/'.venv/bin') + ':' + os.environ['PATH']
os.environ['HF_HOME'] = str(root/'.cache/huggingface')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
subprocess.run(['make','setup'],check=True)
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','libseccomp2'],check=True)
PY=str(root/'.venv/bin/python')
subprocess.run([PY,'-m','pytest','-q'],check=True)


In [ ]:
import datetime
run_id=datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
RUN_ROOT=f'dpo-llm/runs/colab-{run_id}'
WANDB=False
if WANDB:
    subprocess.run([str(root/'.venv/bin/wandb'),'login'],check=True)
flags=['--wandb'] if WANDB else []
def run(script,*args):
    subprocess.run([PY,script,*args,*flags],check=True)
def target(name):
    subprocess.run(['make',name,f'RUN_ROOT={RUN_ROOT}',f'WANDB={"--wandb" if WANDB else ""}'],check=True)


## First results: CartPole and Stage A
Each script prints real metrics when a stage finishes. Tables are generated from CSV only.


In [ ]:
run('marl-ppo/train_single.py','--steps','200000','--out',f'marl-ppo/runs/colab-cartpole-{run_id}')
target('dpo')
run('dpo-llm/eval_rm.py','--adapter',f'{RUN_ROOT}/stage-a/adapter','--out',f'{RUN_ROOT}/eval-rm')


## Code correctness and Stage B
The sandbox must pass its startup checks. If no mixed preferences are found, the run stops explicitly.


In [ ]:
target('code-prefs')
target('stage-b')
for label, adapter in [('base',None),('stage-a',f'{RUN_ROOT}/stage-a/adapter'),('stage-b',f'{RUN_ROOT}/stage-b/adapter')]:
    args=['--label',label,'--out',f'{RUN_ROOT}/eval-{label}']
    if adapter: args+=['--adapter',adapter]
    run('dpo-llm/eval_mbpp.py',*args)


## Remaining CPU experiments (optional in Colab; already supplied locally)
The full grid runs five seeds for each of the four configurations with equal budgets.


In [ ]:
RUN_MARL=False
if RUN_MARL:
    run('marl-ppo/train_single.py','--env','LunarLander-v3','--steps','200000','--out',f'marl-ppo/runs/colab-lunar-{run_id}')
    subprocess.run([PY,'scripts/run_marl_suite.py','--out',f'marl-ppo/colab-{run_id}',*flags],check=True)
    subprocess.run([PY,'marl-ppo/plot.py','--runs',f'marl-ppo/colab-{run_id}'],check=True)
subprocess.run(['make','report'],check=True)
from IPython.display import display, Markdown
display(Markdown(pathlib.Path('dpo-llm/results.md').read_text()))


In [ ]:
# Download all logs and model adapters, excluding dependency/model caches.
import shutil
bundle=pathlib.Path('/content/experiment-results')
if bundle.exists(): shutil.rmtree(bundle)
bundle.mkdir()
for project in ['dpo-llm','marl-ppo']:
    for name in ['runs','plots','results.md']:
        source=root/project/name
        destination=bundle/project/name
        if source.is_dir(): shutil.copytree(source,destination)
        elif source.exists():
            destination.parent.mkdir(parents=True,exist_ok=True)
            shutil.copy2(source,destination)
archive=shutil.make_archive('/content/experiment-results','zip',bundle)
from google.colab import files
files.download(archive)
